In [1]:
import pystac_client
import planetary_computer
import numpy as np

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

In [2]:
grand_canyon = [-112.15, 36.05]
search = catalog.search(
    collections=["cop-dem-glo-30"],
    intersects={"type": "Point", "coordinates": grand_canyon},
)
items = list(search.items())
print(f"Returned {len(items)} items")

Returned 1 items


In [3]:
from pyproj import Transformer

# Create transformer from UTM32 (EPSG:25832) to WGS84
transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)

# Original bbox from DHM (xmin, ymin, xmax, ymax)
xmin, ymin, xmax, ymax = 500000,6175000,550000,6225000

# Convert corners
# lon_min, lat_min = transformer.transform(xmin, ymin)
# lon_max, lat_max = transformer.transform(xmax, ymax)
# lon_min, lat_min = 8.9999, 56.0002
# lon_max, lat_max = 9.9997, 57.0001
lon_min, lat_min, lon_max, lat_max = (9.0, 55.000277777777775, 9.999583333333334, 57.0)

print("WGS84 bbox:")
print(lon_min, lat_min, lon_max, lat_max)

WGS84 bbox:
9.0 55.000277777777775 9.999583333333334 57.0


In [4]:
search = catalog.search(
    collections=["cop-dem-glo-30"],
    bbox=[lon_min, lat_min, lon_max, lat_max],
    query={"gsd": {"eq": 30}}
)
items = list(search.items())

In [5]:
# items[0]

In [6]:
# items[1]

In [7]:
len(items)

2

In [ ]:
import rioxarray
import xarray as xr
from planetary_computer import sign

rasters = []

for item in items:
    signed = sign(item.assets["data"])
    da = rioxarray.open_rasterio(signed.href)
    rasters.append(da)
# signed = sign(items[0].assets["data"])
# da = rioxarray.open_rasterio(signed.href)
# rasters.append(da)

# Merge all tiles
merged = xr.combine_by_coords(rasters)

data = (
    merged.squeeze()
    .drop_vars("band")
    .coarsen({"y": 1, "x": 1})
    .mean()
)

sorted_data = data.sortby("y")

In [10]:
print(data)

<xarray.DataArray (y: 7200, x: 2400)> Size: 69MB
array([[ 0.       ,  0.       ,  0.       , ...,  5.766993 ,  5.1408186,
         4.750998 ],
       [ 0.       ,  0.       ,  0.       , ...,  5.2190013,  5.6950564,
         6.7476044],
       [ 0.       ,  0.       ,  0.       , ...,  5.5977116,  5.8949337,
         6.5014477],
       ...,
       [12.253116 , 12.290995 , 12.52823  , ...,  0.       ,  0.       ,
         0.       ],
       [12.273361 , 12.1914625, 12.147725 , ...,  0.       ,  0.       ,
         0.       ],
       [12.050202 , 12.0864315, 12.488848 , ...,  0.       ,  0.       ,
         0.       ]], shape=(7200, 2400), dtype=float32)
Coordinates:
  * y            (y) float64 58kB 57.0 57.0 57.0 57.0 ... 55.0 55.0 55.0 55.0
  * x            (x) float64 19kB 9.0 9.0 9.001 9.001 ... 9.998 9.999 9.999 10.0
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Point
    scale_factor:   1.0
    add_offset:     0.0


In [ ]:
import xrspatial
from datashader.transfer_functions import shade, stack
from datashader.colors import Elevation

hillshade = xrspatial.hillshade(sorted_data)
stack(shade(hillshade, cmap=["white", "gray"]), shade(sorted_data, cmap=Elevation, alpha=128))

In [ ]:
hillshade.shape

In [ ]:
np.min(hillshade[:,:].coords["x"].values), np.min(hillshade[:,:].coords["y"].values), np.max(hillshade[:,:].coords["x"].values), np.max(hillshade[:,:].coords["y"].values)

In [ ]:
resolution_m = 30
hillshade.shape[0]*resolution_m, hillshade.shape[1]*resolution_m

In [ ]:
from pyproj import Geod
geod = Geod(ellps="WGS84")

xmin, ymin, xmax, ymax = sorted_data.rio.bounds()
lat_mid = (ymin + ymax) / 2
lon_mid = (xmin + xmax) / 2

_, _, width_m = geod.inv(xmin, lat_mid, xmax, lat_mid)
_, _, height_m = geod.inv(lon_mid, ymin, lon_mid, ymax)

print(round(height_m), round(width_m))

In [ ]:
import rioxarray
from pyproj import Geod

# Reproject to UTM zone 32N (ETRS89), with exact 30 m grid
dem_30m = sorted_data.rio.reproject(
    "EPSG:25832",
    resolution=30,
    nodata=sorted_data.rio.nodata
)

print("CRS:", dem_30m.rio.crs)
print("Resolution:", dem_30m.rio.resolution())   # should be close to (30, -30)
print("Shape (rows, cols):", dem_30m.shape)

# Dimensions from pixel count
h_m = dem_30m.shape[0] * 30
w_m = dem_30m.shape[1] * 30
print("Pixel-based size (m):", h_m, w_m)

# Optional: dimensions from projected bounds (more exact on edges)
xmin, ymin, xmax, ymax = dem_30m.rio.bounds()
print("Bounds-based size (m):", ymax - ymin, xmax - xmin)

In [ ]:
dem_flip_y = dem_30m.isel(y=slice(None, None, -1))
hillshade = xrspatial.hillshade(dem_flip_y)
stack(shade(hillshade, cmap=["white", "gray"]), shade(dem_flip_y, cmap=Elevation, alpha=128))